# AreaSqKm_01_Notebook
Javier Parada

October 13, 2021

## Purpose
This notebook documents the procedure followed in the measurement of areas. 

In [5]:
# Data manipulation
import pandas as pd
import numpy as np
pd.options.display.float_format = "{:,.2f}".format

# Options for pandas
pd.options.display.max_columns = 50
pd.options.display.max_rows = 50

# Visualizations
import plotly
import plotly.graph_objs as go
import plotly.offline as ply
plotly.offline.init_notebook_mode(connected=True)

import plotly.express as px

import cufflinks as cf
cf.go_offline(connected=True)
cf.set_config_file(theme='white')

import matplotlib as plt

# Autoreload extension
if 'autoreload' not in get_ipython().extension_manager.loaded:
    %load_ext autoreload
    
%autoreload 2

In [10]:
# Additional libraries

#import glob
import random
import folium
import wbgapi as wb
import geopandas as gpd
import matplotlib.pyplot as plt
#import plotly.graph_objects as go

import ee
ee.Initialize()
import leafmap
#leafmap.update_package()
import geemap
#geemap.update_package()

OSError: Could not find lib c or load any of its variants [].

In [7]:
out_dir = os.path.join(os.path.expanduser('~'), 'Documents/Websites/JavierParada.github.io/EL5M/')
print(out_dir)

/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/


In [8]:
gaul0 = ee.FeatureCollection("FAO/GAUL/2015/level0")
gaul1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
gaul2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")

NameError: name 'ee' is not defined

In [89]:
GAUL_2_3411 = 'shapes/districts.shp'
GAUL_borders_2 = gpd.read_file(os.path.join(out_dir,GAUL_2_3411))

In [92]:
borders_0 = pd.DataFrame(pd.unique(GAUL_borders_2["ADM0_CODE"]))
borders_1 = pd.DataFrame(pd.unique(GAUL_borders_2["ADM1_CODE"]))
borders_2 = pd.DataFrame(pd.unique(GAUL_borders_2["ADM2_CODE"]))
print('{} countries, {} states, {} provinces in WDI'.format(len(borders_0), len(borders_1), len(GAUL_borders_2)))

255 countries, 3409 states, 38229 provinces in WDI


In [80]:
Map = geemap.Map(center = (40, -100), zoom = 3)
Map

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(T…

In [81]:
ee_class_table = """
Value	Color	Description
1	05450a	Evergreen Needleleaf Forests: dominated by evergreen conifer trees (canopy >2m). Tree cover >60%.
2	086a10	Evergreen Broadleaf Forests: dominated by evergreen broadleaf and palmate trees (canopy >2m). Tree cover >60%.
3	54a708	Deciduous Needleleaf Forests: dominated by deciduous needleleaf (larch) trees (canopy >2m). Tree cover >60%.
4	78d203	Deciduous Broadleaf Forests: dominated by deciduous broadleaf trees (canopy >2m). Tree cover >60%.
5	009900	Mixed Forests: dominated by neither deciduous nor evergreen (40-60% of each) tree type (canopy >2m). Tree cover >60%.
6	c6b044	Closed Shrublands: dominated by woody perennials (1-2m height) >60% cover.
7	dcd159	Open Shrublands: dominated by woody perennials (1-2m height) 10-60% cover.
8	dade48	Woody Savannas: tree cover 30-60% (canopy >2m).
9	fbff13	Savannas: tree cover 10-30% (canopy >2m).
10	b6ff05	Grasslands: dominated by herbaceous annuals (<2m).
11	27ff87	Permanent Wetlands: permanently inundated lands with 30-60% water cover and >10% vegetated cover.
12	c24f44	Croplands: at least 60% of area is cultivated cropland.
13	a5a5a5	Urban and Built-up Lands: at least 30% impervious surface area including building materials, asphalt and vehicles.
14	ff6d4c	Cropland/Natural Vegetation Mosaics: mosaics of small-scale cultivation 40-60% with natural tree, shrub, or herbaceous vegetation.
15	69fff8	Permanent Snow and Ice: at least 60% of area is covered by snow and ice for at least 10 months of the year.
16	f9ffa4	Barren: at least 60% of area is non-vegetated barren (sand, rock, soil) areas with less than 10% vegetation.
17	1c0dff	Water Bodies: at least 60% of area is covered by permanent water bodies.

"""
legend_dict = geemap.legend_from_ee(ee_class_table)
Map.add_legend(legend_title="MODIS Global Land Cover", legend_dict=legend_dict)

In [84]:
roi = gaul2.filter(ee.Filter.eq('ADM0_NAME', 'India'))
Map.addLayer(roi, {'color': '00909F', 'fillColor': 'b5ffb4', 'width': .1}, 'Second Level Administrative Units')
Map.centerObject(roi)

In [43]:
# Get the MODIS Yearly 500m Landcover Collection
modisLandCover = ee.Image("MODIS/006/MCD12Q1/2018_01_01")
igbpLandCover = modisLandCover.select('LC_Type1')
igbpLandCoverVis = {
  'min': 1.0,
  'max': 17.0,
  'palette': [
    '05450a', '086a10', '54a708', '78d203', '009900', 'c6b044', 'dcd159',
    'dade48', 'fbff13', 'b6ff05', '27ff87', 'c24f44', 'a5a5a5', 'ff6d4c',
    '69fff8', 'f9ffa4', '1c0dff'
  ],
}
Map.addLayer(igbpLandCover.clip(roi), igbpLandCoverVis, 'IGBP Land Cover')

In [44]:
urban = igbpLandCover.eq(13)
Map.addLayer(urban.selfMask().clip(roi), {'min': 0, 'max':1, 'palette': ['grey','black']}, 'Urban')

In [45]:
merit = ee.Image("MERIT/DEM/v1_0_3").select('dem')
lowAreas = merit.lte(5)
Map.addLayer(lowAreas.selfMask().clip(roi), {'min': 0, 'max':1, 'palette': ['grey','blue']}, "Low Areas")

In [46]:
out_dir = os.path.join(os.path.expanduser('~'), 'Downloads')
nlcd_stats = os.path.join(out_dir, 'kerala.csv')  
geemap.zonal_statistics_by_group(igbpLandCover.clip(roi), 
                                 roi, 
                                 nlcd_stats, 
                                 statistics_type='SUM', 
                                 denominator=1000000, 
                                 decimal_places=2,
                                 scale = 500)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Downloads/kerala.csv


In [47]:
df = wb.data.DataFrame(['AG.LND.EL5M.ZS', 'EN.POP.EL5M.ZS'], wb.region.members('WLD'), time=2010, labels=True)
df = df.sort_values(by=['Country'], ascending=True).reset_index()
df["economy"]= df["economy"].replace("XKX", "KSV") # replace code for Kosovo
df = df[df["economy"]!="CHI"] # remove Channel Islands
countries = list(df["economy"])
print('{} countries in WDI'.format(len(countries)))
df

216 countries in WDI


,economy,Country,AG.LND.EL5M.ZS,EN.POP.EL5M.ZS
0,AFG,Afghanistan,NaN,NaN
1,ALB,Albania,4.86,7.07
2,DZA,Algeria,0.03,0.76
3,ASM,American Samoa,3.56,3.27
4,AND,Andorra,NaN,NaN
...,...,...,...,...
212,VIR,Virgin Islands (U.S.),6.64,5.44
213,PSE,West Bank and Gaza,0.09,0.55
214,YEM,"Yemen, Rep.",0.41,1.58
215,ZMB,Zambia,NaN,NaN


In [ ]:
df[